# Analiza Statystyczna Kaskady TBI

W pierwszej sekcji przeprowadzona zostanie analiza podgrup klinicznych, weryfikująca czy model nie faworyzuje konkretnych demografii. W drugiej sekcji wygenerowane zostaną wykresy zależności SHAP, obrazujące nieliniowe wnioskowanie systemu.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.metrics import roc_auc_score, recall_score, balanced_accuracy_score

# Ustawienia stylu dla wydruku
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

## Analiza Podgrup

Ewaluacja metryk modelu (AUC, Zbalansowana Dokładność, Czułość) w izolowanych, krytycznych podgrupach medycznych. Sprawdzamy m.in. zachowanie modelu przy pacjentach zaintubowanych (gdzie skala GCS jest maskowana) oraz w hipoperfuzji. Poniższy kod oblicza metryki i zapisuje je do pliku CSV, gotowego do importu w środowisku LaTeX.

In [ ]:
print("Generowanie analizy podgrup.")

    # Definiujemy słownik z indeksami pacjentów dla poszczególnych podgrup z X_test
    subgroups_raw = {
        "Wszyscy Pacjenci": X_test.index,
        "Seniorzy (Wiek >= 65)": X_test[X_test['age'] >= 65].index,
        "Młodsi (Wiek < 65)": X_test[X_test['age'] < 65].index,
        "Lekkie TBI (GCS >= 11)": X_test[X_test['gcs_sum'] >= 11].index,
        "Ciężkie TBI (GCS <= 8)": X_test[X_test['gcs_sum'] <= 8].index,
        "Zaintubowani (Maskowany GCS)": X_test[X_test['is_intubated'] == 1].index,
        "Wstrząs / Hipoperfuzja (MAP < 65)": X_test[X_test['map_min'] < 65].index if 'map_min' in X_test.columns else X_test[X_test['sysbp_min'] < 90].index,
        # Weryfikacja skuteczności inżynierii cech
        "Obciążeni Neurologicznie (Deficyt)": X_test[X_test['baseline_neuro_deficit'] == 1].index if 'baseline_neuro_deficit' in X_test.columns else [],
        "Myląca Sedacja": X_test[X_test['gcs_sedation_mismatch'] == 1].index if 'gcs_sedation_mismatch' in X_test.columns else []
    }

    # Zabezpieczenie: odrzucamy podgrupy bez pacjentów w danym wycinku X_test
    subgroups = {k: v for k, v in subgroups_raw.items() if len(v) > 0}

    results = []

for group_name, indices in subgroups.items():
    if len(indices) == 0:
        continue
        
    # Ekstrakcja danych dla danej podgrupy (używamy indeksów z głównego zbioru)
    y_test_sub = y_test.loc[indices]
    y_prob_sub = y_final_prob[X_test.index.get_indexer(indices)]
    y_pred_sub = y_threshold_pred[X_test.index.get_indexer(indices)]
    
    # Obliczanie metryk (AUC wymaga co najmniej 1 próbki z obu klas)
    try:
        auc = roc_auc_score(y_test_sub, y_prob_sub) if len(np.unique(y_test_sub)) > 1 else np.nan
    except ValueError:
        auc = np.nan
        
    recall = recall_score(y_test_sub, y_pred_sub, zero_division=0)
    bal_acc = balanced_accuracy_score(y_test_sub, y_pred_sub)
    
    results.append({
        "Podgrupa Kliniczna": group_name,
        "Liczba Pacjentów": len(indices),
        "Zgony": y_test_sub.sum(),
        "AUC": f"{auc:.3f}" if not np.isnan(auc) else "N/A",
        "Zbalansowana Dokładność": f"{bal_acc:.3f}",
        "Czułość (Recall)": f"{recall:.3f}"
    })

df_subgroups = pd.DataFrame(results)

# Zapis tabeli do CSV, aby łatwo przenieść dane do LaTeXa
df_subgroups.to_csv('models/subgroup_analysis.csv', index=False)
print("\n Analiza Podgrup")
print(df_subgroups.to_string(index=False))

### Wizualizacja wyników podgrup

Wykres słupkowy ilustrujący stabilność parametru AUC we wszystkich wyodrębnionych podgrupach w stosunku do ogólnego wyniku kaskady.

In [ ]:
# Wizualizacja AUC w podgrupach
df_plot = df_subgroups.iloc[1:].copy()
df_plot['AUC'] = pd.to_numeric(df_plot['AUC'], errors='coerce')

plt.figure(figsize=(10, 5))
ax = sns.barplot(x='AUC', y='Podgrupa Kliniczna', data=df_plot, palette='viridis')
plt.axvline(x=roc_auc_score(y_test, y_final_prob), color='red', linestyle='--', label='Ogólne AUC Modelu')
plt.title('Wydajność Kaskady (AUC) w izolowanych podgrupach klinicznych', fontweight='bold')
plt.xlim(0.5, 1.0)
plt.legend(loc='lower left')
plt.tight_layout()
plt.savefig('models/subgroup_auc_plot.png', dpi=300)
plt.show()

## Krzywe Zależności SHAP

Analiza obrazująca, w jaki sposób zmiana wartości konkretnego parametru klinicznego wpływa na ostateczne ryzyko zgonu wyliczone przez system. Umożliwia to zbadanie, czy model wyłapał poprawne fizjologicznie, nieliniowe zależności.

In [ ]:
print("Generowanie wykresów zależności SHAP dla pełnej Kaskady MoE.")

    # Zmieniamy układ na 3 wykresy obok siebie
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(22, 6))

    # Dla KernelExplainer shap_values_cascade bywa listą, pobieramy indeks 0 dla klasy pozytywnej
    shap_vals = shap_values_cascade[0] if isinstance(shap_values_cascade, list) else shap_values_cascade

    # Wykres 1: GCS Sum dla całej kaskady
    shap.dependence_plot(
        "gcs_sum", 
        shap_vals, 
        test_sample, 
        ax=ax1, 
        show=False, 
        interaction_index="age" # Pokazuje interakcję z wiekiem
    )
    ax1.set_title('Wpływ wyniku GCS na decyzję o zgonie', fontweight='bold', fontsize=13)
    ax1.set_xlabel('Punktacja GCS', fontweight='bold')
    ax1.set_ylabel('Wpływ SHAP (ryzyko zgonu)', fontweight='bold')

    # Wykres 2: Średnie ciśnienie tętnicze (map_min)
    if 'map_min' in test_sample.columns:
        shap.dependence_plot(
            "map_min", 
            shap_vals, 
            test_sample, 
            ax=ax2, 
            show=False, 
            interaction_index=None
        )
        ax2.set_title('Wpływ Hipoperfuzji (MAP) na decyzję', fontweight='bold', fontsize=13)
        ax2.set_xlabel('Najniższe Średnie Ciśnienie Tętnicze (MAP)', fontweight='bold')
        ax2.set_ylabel('Wpływ SHAP', fontweight='bold')
        ax2.axvline(x=65, color='red', linestyle='--', alpha=0.6, label='Próg ciężkiej hipoperfuzji')
        ax2.legend()
    else:
        # Fallback na skurczowe ciśnienie krwi
        shap.dependence_plot("sysbp_min", shap_vals, test_sample, ax=ax2, show=False, interaction_index=None)
        ax2.set_title('Wpływ najniższego ciśnienia skurczowego', fontweight='bold', fontsize=13)

    # Wykres 3: Zależność SHAP od strefy niepewności Konsylium
    if 'model_1_prob' in test_sample.columns:
        shap.dependence_plot(
            "model_1_prob", 
            shap_vals, 
            test_sample, 
            ax=ax3, 
            show=False, 
            interaction_index=None
        )
        ax3.set_title('Wpływ strefy niepewności na końcowy wynik', fontweight='bold', fontsize=13)
        ax3.set_xlabel('Prawdopodobieństwo bazowe Konsylium', fontweight='bold')
        # Dodajemy tło zaznaczające strefę aktywacji Turnieju Specjalistów
        ax3.axvspan(0.35, 0.65, color='orange', alpha=0.15, label='Aktywacja Turnieju Fenotypowego')
        ax3.legend(loc='upper left')

    plt.tight_layout()
    plt.savefig('models/shap_dependence_plots_cascade.png', dpi=300, bbox_inches='tight')
    print("Zapisano: models/shap_dependence_plots_cascade.png")
    plt.show()